# 📘 FULL PIPELINE NOTEBOOK — GPT‑4o Vision OCR → Embeddings → Auto‑Labels → ML Scoring → GPT‑4o Reasoning
### Option A1 — Financial Section Classification

In [ ]:
!pip install openai pandas tqdm pyarrow scikit-learn joblib requests

In [ ]:
from openai import OpenAI
import pandas as pd, numpy as np, time, os, joblib, requests
from tqdm import tqdm

OPENAI_API_KEY = 'YOUR_OPENAI_KEY'
client = OpenAI(api_key=OPENAI_API_KEY)

## Step 1 — Load Dataset

In [ ]:
sas_url = 'https://<yourstorage>.blob.core.windows.net/multifinben/raw/train-00000-of-00008.parquet?<sas>'
df = pd.read_parquet(sas_url)
df.head()

## Step 2 — GPT‑4o Vision OCR

In [ ]:
def gpt4o_vision_ocr(base64_str):
    resp = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[{ 'role':'user', 'content':[
            {'type':'text','text':'Extract ONLY the text from this image.'},
            {'type':'image_url','image_url':{'url': f'data:image/png;base64,{base64_str}'}}]}])
    return resp.choices[0].message.content

## Step 3 — Budget‑Safe OCR Loop

In [ ]:
MAX_SPEND_USD = 8.0
COST = 0.005
OUT = 'ocr.pkl'
if 'ocr_text' not in df.columns: df['ocr_text']=None
if os.path.exists(OUT): df.update(pd.read_pickle(OUT))
spent=0; processed=0
for i in tqdm(range(len(df))):
    if isinstance(df.loc[i,'ocr_text'],str) and len(df.loc[i,'ocr_text'])>1: continue
    if spent + COST > MAX_SPEND_USD: break
    try:
        t = gpt4o_vision_ocr(df.loc[i,'image'])
        df.loc[i,'ocr_text']=t
        spent+=COST; processed+=1
        if processed%20==0: df.to_pickle(OUT)
        time.sleep(0.3)
    except Exception as e: print('Err:',e); time.sleep(1)
df.to_pickle(OUT); print('OCR done:',processed)

## Step 4 — OpenAI Embeddings

In [ ]:
def embed_text(t):
    if t is None or len(t.strip())==0: return None
    r = client.embeddings.create(model='text-embedding-3-large', input=t)
    return r.data[0].embedding

In [ ]:
df['embedding'] = [embed_text(t) for t in tqdm(df['ocr_text'])]
df.to_pickle('emb.pkl')
df.head()

## Step 5 — GPT‑4o Auto‑Labeling (A1 Financial Sections)

In [ ]:
CATEGORIES = ['Risk','Financials','Management','Operations','Legal','Other']

def auto_label(text):
    if text is None or len(text.strip())==0: return 'Other'
    resp = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[
            {'role':'system','content': 'Classify the SEC filing page into ONE category: ' + ', '.join(CATEGORIES)},
            {'role':'user','content': text}
        ]
    )
    label = resp.choices[0].message.content.strip()
    return label if label in CATEGORIES else 'Other'

In [ ]:
df['label'] = [auto_label(t) for t in tqdm(df['ocr_text'])]
df['label'].value_counts()

## Step 6 — Encode Labels

In [ ]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df['label_id'] = le.fit_transform(df['label'])
df[['label','label_id']].head()

## Step 7 — Train ML Classifier

In [ ]:
df_train = df[df['embedding'].apply(lambda x: isinstance(x,list) and len(x)==3072)].reset_index(drop=True)
X = np.vstack(df_train['embedding'])
y = df_train['label_id']
from sklearn.linear_model import LogisticRegression
model = LogisticRegression(max_iter=2000,class_weight='balanced')
model.fit(X,y)
joblib.dump(model,'fusion_model.joblib')
joblib.dump(le,'label_encoder.joblib')
model

## Step 8 — Azure ML score.py Script

In [ ]:
%%writefile score.py
import json, numpy as np, joblib, os
def init():
    global model, le
    path = os.environ['AZUREML_MODEL_DIR']
    model = joblib.load(os.path.join(path,'fusion_model.joblib'))
    le = joblib.load(os.path.join(path,'label_encoder.joblib'))
def run(raw):
    d = json.loads(raw)
    emb = np.array(d['embedding'])
    pred = model.predict([emb])[0]
    score = float(model.predict_proba([emb])[0].max())
    label = le.inverse_transform([pred])[0]
    return {'predicted_class': label, 'score': score}

## Step 9 — Final Pipeline Output

In [ ]:
def process_page(index, use_endpoint=False, endpoint_url=None, endpoint_key=None):
    res = {}
    text = df.loc[index,'ocr_text']
    emb = df.loc[index,'embedding']
    res['ocr_text'] = text
    res['embedding'] = emb
    if use_endpoint and endpoint_url:
        headers={'Authorization':f'Bearer {endpoint_key}'}
        r = requests.post(endpoint_url,json={'embedding':emb},headers=headers)
        try:
            j=r.json()
            res['predicted_class']=j.get('predicted_class')
            res['score']=j.get('score')
        except: res['score']=None; res['predicted_class']=None
    else:
        res['score']=None; res['predicted_class']=None
    # GPT-4o reasoning
    resp = client.chat.completions.create(
        model='gpt-4o',
        messages=[{'role':'system','content':'You are a financial analyst.'},
                 {'role':'user','content': f'''Summarise and explain this SEC filing page:
{text}'''}])
    summary = resp.choices[0].message.content
    res['gpt4_summary'] = summary
    res['gpt4_explanation'] = summary
    return res

### Run pipeline for page 0

In [ ]:
process_page(0)